## Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn import tree

from models import LogisticRegression, DecisionTree

## Data reading

In [2]:
data = pd.read_csv("data.csv") 
data.head()
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## Data Preprocessing

In [3]:
data.drop("customerID", axis=1, inplace=True) # Drop customerID as it is not useful for prediction
data.head() # Check the first few rows of the data after dropping customerID

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Check for any null values and space skipped values on totalcharges

In [4]:
data['TotalCharges'].isnull().sum() # Check for null values in TotalCharges
data[data['TotalCharges'] == ' '] # Check for space skipped values in TotalCharges

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


In [5]:
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce') # Convert TotalCharges to numeric, coercing errors to NaN
data['TotalCharges'].isnull().sum() # Check for any null values in TotalCharges

11

Correct Nan values with median, 
actually we have several options, 1. fill with 0, 2. drop row, 3. fill with median
Since models cannot work with None values, there must be something
So why 3rd option, its not the perfect option, but least risky
First option is really bad idea as it will destroy our model as missing data is not 0 value
Second option safe but it makes dataset smaller, you can play with this, here we will do with third option, and compare with this as there are only 11 rows with unknown totalCharge, but if 3000 out of 10000 is unknown dont drop rows
and finally third option, also affects out model learning but slightly

In [6]:
data['TotalCharges'] = data['TotalCharges'].fillna(data['TotalCharges'].median()) # Fill NaN values in TotalCharges with the median
data['TotalCharges'].isnull().sum() # Check again for any null values in TotalCharges, so we have no null values in totalCharges now

0

In [7]:
print(data['Churn'].unique())
print(data['Churn'].isnull().sum()) # Check for null values in Churn column
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0, 1: 1, 0: 0}) # Convert Churn column to binary values 
print(data['Churn'].value_counts()) # Check the distribution of Churn values


['No' 'Yes']
0
Churn
0    5174
1    1869
Name: count, dtype: int64


In [8]:
data['SeniorCitizen'].info()
data['SeniorCitizen'] = data['SeniorCitizen'].astype('object') # Convert SeniorCitizen to object type


<class 'pandas.core.series.Series'>
RangeIndex: 7043 entries, 0 to 7042
Series name: SeniorCitizen
Non-Null Count  Dtype
--------------  -----
7043 non-null   int64
dtypes: int64(1)
memory usage: 55.1 KB


## Splitting the dataset

In [9]:
X = data.drop('Churn', axis=1) # Features
y = data['Churn'] # Target variable
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # Split the dataset into training and testing sets
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape) # Check the shapes of the training and testing sets



(5634, 19) (1409, 19) (5634,) (1409,)


## encode

In [10]:
# Check for categorical columns
categorical = list(X_train.select_dtypes(include=['object']).columns)
print(categorical)
# One-hot encode the categorical columns
X_train = pd.get_dummies(X_train, columns=categorical, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical, drop_first=True)
X_train.head() # Check the first few rows of the data after encoding categorical variables

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


,tenure,MonthlyCharges,TotalCharges,gender_Male,SeniorCitizen_1,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
2142,21,64.85,1336.800,False,False,False,True,True,False,False,...,False,False,False,True,True,False,False,False,False,True
1623,54,97.20,5129.450,False,False,False,False,True,False,True,...,False,True,False,True,False,True,True,False,False,False
6074,1,23.45,23.450,True,False,True,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
1362,4,70.20,237.950,True,False,False,False,True,False,False,...,False,False,False,False,False,False,True,False,True,False
6754,0,61.90,1397.475,True,False,False,True,True,False,True,...,False,False,False,False,False,True,True,False,False,False


## Models

In [11]:
# Decision Tree Classifier
model1 = DecisionTreeClassifier(criterion="entropy", max_depth=4)
model1.fit(X_train, y_train)
y_pred_dt = model1.predict(X_test)
print("Decision Tree Classifier:")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("Classification Report:\n", classification_report(y_test, y_pred_dt))



Decision Tree Classifier:
Accuracy: 0.7955997161107168
Confusion Matrix:
 [[975  61]
 [227 146]]
Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.94      0.87      1036
           1       0.71      0.39      0.50       373

    accuracy                           0.80      1409
   macro avg       0.76      0.67      0.69      1409
weighted avg       0.78      0.80      0.77      1409



## standardize

In [12]:
numerical = list(X_train.select_dtypes(include=['number']).columns)
print(numerical)
scaler = StandardScaler()
X_train[numerical] = scaler.fit_transform(X_train[numerical])
X_test[numerical] = scaler.transform(X_test[numerical])


['tenure', 'MonthlyCharges', 'TotalCharges']


In [13]:
# Logistic Regression
X_train.dtypes.unique()
model2 = LogisticRegression(n_iters=1000)
model2.fit(X_train, y_train)
y_pred_lr = model2.predict(X_test)
print("Logistic Regression:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("Classification Report:\n", classification_report(y_test, y_pred_lr))


Logistic Regression:
Accuracy: 0.808374733853797
Confusion Matrix:
 [[956  80]
 [190 183]]
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.92      0.88      1036
           1       0.70      0.49      0.58       373

    accuracy                           0.81      1409
   macro avg       0.77      0.71      0.73      1409
weighted avg       0.80      0.81      0.80      1409



We scaled only after calculated the decision tree one, since for decision tree scaling is not good choice, but encoded both for decision and logistic

I hope you enjoyed you can play by changing data preprocessing and/or scaling, encoding mehtods (may be LabelEncoding)